# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkhlor006/Flyrank_internship_ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/mkhlor006/Flyrank_internship_ML.git

Cloning into 'Flyrank_internship_ML'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 142 (delta 51), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.86 MiB | 13.25 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [2]:
import os

os.chdir("/content/Flyrank_internship_ML")

print("Current directory:")
print(os.getcwd())

print("\nTop-level files/folders:")
print(os.listdir("."))

Current directory:
/content/Flyrank_internship_ML

Top-level files/folders:
['scripts', 'requirements.txt', 'data', '02_your_first_readable_model.ipynb', 'GUIDE.md', 'work', '.github', 'AGENTS.md', 'outputs', 'CLAUDE.md', 'README.md', 'LICENSE', '01_first_look_and_discovery.ipynb', 'submission', '.git', '.gitignore', 'notebooks', 'DATA_USE.md', 'docs', 'SETUP.md', 'skills']


In [3]:
import os

print(os.path.exists("scripts/01_prepare_features.py"))

True


In [6]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/Flyrank_internship_ML/data/processed/refresh_feature_vector.csv


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

My lane is a classification and ranking problem: I want to identify content pages that may need refresh review, with priority given to the pages most likely to be declining.

I will compare three supervised classification methods: Logistic Regression, a Decision Tree, and a Random Forest. Logistic Regression provides a simple and interpretable baseline for a learned model. A Decision Tree is useful because its rules are easy to inspect. Random Forest can capture non-linear relationships and interactions between multiple page characteristics.

Because the practical decision is which pages should be reviewed first, I evaluate the models using Precision@50 as the primary metric. This measures how many of the top 50 ranked pages are actually declining. I will also report supporting metrics such as precision, recall, F1, ROC-AUC and average precision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
import pandas as pd

feature_path = "data/processed/refresh_feature_vector.csv"
data = pd.read_csv(feature_path)

print("Prepared data shape:", data.shape)
print("Target counts:")
print(data["is_declining_label"].value_counts())
print("Clients:", data["client_id"].nunique())

Prepared data shape: (30000, 52)
Target counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Clients: 32


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

clients = data["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train = data[data["client_id"].isin(train_clients)].copy()
test = data[data["client_id"].isin(test_clients)].copy()

print("Total clients:", len(clients))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nTraining rows:", len(train))
print("Test rows:", len(test))

print("\nTraining class counts:")
print(train["is_declining_label"].value_counts())

print("\nTest class counts:")
print(test["is_declining_label"].value_counts())

Total clients: 32
Training clients: 25
Test clients: 7

Training rows: 26581
Test rows: 3419

Training class counts:
is_declining_label
1    14471
0    12110
Name: count, dtype: int64

Test class counts:
is_declining_label
1    1791
0    1628
Name: count, dtype: int64


In [10]:
overlap = set(train["client_id"]) & set(test["client_id"])
print("Client overlap:", len(overlap))

Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [13]:
!python scripts/02_baseline_score.py

Wrote baseline queue: /content/Flyrank_internship_ML/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


In [14]:
import os

print(
    "Baseline file exists:",
    os.path.exists("data/processed/baseline_refresh_queue.csv")
)

Baseline file exists: True


In [15]:
baseline_df = pd.read_csv(
    "data/processed/baseline_refresh_queue.csv"
)

print("Baseline shape:", baseline_df.shape)
print("Baseline columns:", baseline_df.columns.tolist())
print(baseline_df.head())

Baseline shape: (30000, 22)
Baseline columns: ['content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score', 'visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score', 'reason_codes', 'suggested_action_baseline', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction']
             content_id          client_id  baseline_rank  \
0  content_9532f197bbc8  client_4e07408562              1   
1  content_4d1fe5b32dc2  client_19581e27de              2   
2  content_07f2e7a6f38a  client_19581e27de              3   
3  content_e5ae436f9a16  client_4e07408562              4   
4  content_3430a8b94511  client_19581e27de              5   

   baseline_refresh_score  visibility_score  freshness_risk_score  \
0                0.941189          0.999633                0.8432   
1                0.934889        

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

# Keep only feature columns that are actually present.
numeric_features = [
    c for c in NUMERIC_FEATURES if c in data.columns
]

categorical_features = [
    c for c in CATEGORICAL_FEATURES if c in data.columns
]

# Numeric features
train_numeric = (
    train[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

test_numeric = (
    test[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Categorical features
train_cat = train[categorical_features].fillna("unknown").astype(str)
test_cat = test[categorical_features].fillna("unknown").astype(str)

# One-hot encoding
train_cat_encoded = pd.get_dummies(
    train_cat,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float
)

test_cat_encoded = pd.get_dummies(
    test_cat,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float
)

# Make train/test columns identical.
test_cat_encoded = test_cat_encoded.reindex(
    columns=train_cat_encoded.columns,
    fill_value=0
)

X_train = pd.concat(
    [
        train_numeric.reset_index(drop=True),
        train_cat_encoded.reset_index(drop=True)
    ],
    axis=1
)

X_test = pd.concat(
    [
        test_numeric.reset_index(drop=True),
        test_cat_encoded.reset_index(drop=True)
    ],
    axis=1
)

y_train = train["is_declining_label"].astype(int).reset_index(drop=True)
y_test = test["is_declining_label"].astype(int).reset_index(drop=True)

print("Training feature matrix:", X_train.shape)
print("Test feature matrix:", X_test.shape)
print("Number of model features:", X_train.shape[1])

Training feature matrix: (26581, 52)
Test feature matrix: (3419, 52)
Number of model features: 52


In [17]:
baseline_df = pd.read_csv(
    "data/processed/baseline_refresh_queue.csv"
)

print(baseline_df.shape)
print(baseline_df.columns.tolist())

(30000, 22)
['content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score', 'visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score', 'reason_codes', 'suggested_action_baseline', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction']


In [18]:
print(baseline_df.shape)
print(baseline_df.columns.tolist())

(30000, 22)
['content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score', 'visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score', 'reason_codes', 'suggested_action_baseline', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction']


In [20]:
# Safely inspect the variables currently in the notebook

import pandas as pd

items = list(globals().items())

found = False

for name, obj in items:
    if isinstance(obj, pd.DataFrame):
        cols = [str(c).lower() for c in obj.columns]

        keywords = [
            "model", "precision", "recall",
            "f1", "roc", "accuracy", "baseline"
        ]

        if any(
            any(k in col for k in keywords)
            for col in cols
        ):
            found = True
            print(f"\n--- DataFrame: {name} ---")
            print("Shape:", obj.shape)
            print("Columns:", list(obj.columns))
            display(obj)

if not found:
    print("No model-comparison DataFrame was found in memory.")


--- DataFrame: data ---
Shape: (30000, 52)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.000000,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.000000,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.000000,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.000000,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.000000,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,content_c322796023c8,client_e29c9c180c,10.0,0.05,LOW,0.00,keyword article,transactional,1386.0,9084.0,...,new,0.0,0,0.693147,0.000000,1.386294,0.000000,0,0,0
29996,content_526572edb3fa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2654.0,17056.0,...,down,-75.1,1,6.635947,1.386294,1.386294,0.000000,1,0,1
29997,content_38112bdd0c6e,client_349c41201b,10.0,1.00,HIGH,0.00,keyword article,transactional,2857.0,18725.0,...,down,-66.2,1,8.754161,2.564949,3.806662,0.000000,1,0,1
29998,content_ab26273a7e7a,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,0.0,0.0,...,down,-27.9,1,11.949657,5.831882,6.139885,0.000000,1,0,1



--- DataFrame: train ---
Shape: (26581, 52)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.000000,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.000000,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.000000,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.000000,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.000000,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,content_c322796023c8,client_e29c9c180c,10.0,0.05,LOW,0.00,keyword article,transactional,1386.0,9084.0,...,new,0.0,0,0.693147,0.000000,1.386294,0.000000,0,0,0
29996,content_526572edb3fa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2654.0,17056.0,...,down,-75.1,1,6.635947,1.386294,1.386294,0.000000,1,0,1
29997,content_38112bdd0c6e,client_349c41201b,10.0,1.00,HIGH,0.00,keyword article,transactional,2857.0,18725.0,...,down,-66.2,1,8.754161,2.564949,3.806662,0.000000,1,0,1
29998,content_ab26273a7e7a,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,0.0,0.0,...,down,-27.9,1,11.949657,5.831882,6.139885,0.000000,1,0,1



--- DataFrame: test ---
Shape: (3419, 52)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
13,content_a5a2fbc76336,client_8527a891e2,10.0,0.00,LOW,0.00,keyword article,informational,1342.0,8469.0,...,stable,10.4,0,5.730100,0.000000,1.609438,0.0,0,0,1
15,content_689414059706,client_9f14025af0,0.0,0.00,LOW,0.00,keyword article,informational,2661.0,17889.0,...,up,675.0,0,3.663562,2.302585,2.302585,0.0,1,0,0
32,content_5eeba5d398f2,client_bbb965ab0c,0.0,0.00,LOW,0.00,keyword article,informational,2949.0,21532.0,...,down,-40.0,1,6.410175,1.609438,2.833213,0.0,1,0,1
39,content_4595e8704e07,client_8527a891e2,90.0,0.06,LOW,0.03,keyword article,informational,3666.0,21824.0,...,down,-100.0,1,1.609438,0.000000,0.693147,0.0,0,0,0
40,content_4df0b7207fe3,client_9f14025af0,10.0,0.00,LOW,0.00,keyword article,transactional,896.0,6300.0,...,up,197.7,0,7.438972,3.367296,3.465736,0.0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29950,content_7d52e164c23c,client_8527a891e2,0.0,0.00,LOW,0.00,keyword article,informational,1644.0,10494.0,...,down,-50.0,1,2.397895,0.000000,2.197225,0.0,0,0,0
29955,content_a31cad37ea8a,client_8b940be7fb,10.0,0.00,LOW,0.00,keyword article,commercial,3269.0,20316.0,...,stable,-9.2,0,8.709300,3.850148,2.944439,0.0,1,0,1
29961,content_326a540b3a6e,client_8527a891e2,20.0,0.00,LOW,0.00,keyword article,commercial,3919.0,24518.0,...,down,-79.2,1,3.526361,0.000000,1.098612,0.0,0,0,0
29985,content_fcad8c75dc44,client_bbb965ab0c,10.0,0.00,LOW,0.00,keyword article,informational,2666.0,19294.0,...,stable,6.7,0,8.396606,3.583519,4.564348,0.0,1,0,1



--- DataFrame: baseline_df ---
Shape: (30000, 22)
Columns: ['content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score', 'visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score', 'reason_codes', 'suggested_action_baseline', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction']


,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.843200,0.979233,0.871347,declining_with_demand|page_one_decay_risk|low_...,refresh,...,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.843200,0.963733,0.866582,page_one_decay_risk|low_engagement_visible_page,monitor,...,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.843200,0.959965,0.866843,page_one_decay_risk|low_engagement_visible_page,monitor,...,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.843200,0.955347,0.868180,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.843200,0.951314,0.870069,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,content_994b0a4e4dde,client_98a3ab7c34,29996,0.008241,0.017933,0.001983,0.000000,0.009456,general_refresh_review,monitor,...,0,2,0.0,0.00,0.00,50.00,124,1,2532.0,flat
29996,content_0a22a2eeefdd,client_98a3ab7c34,29997,0.008216,0.017933,0.001983,0.000000,0.008946,general_refresh_review,monitor,...,0,2,0.0,0.00,0.00,50.00,125,1,2608.0,new
29997,content_5168e96834b9,client_98a3ab7c34,29998,0.008081,0.017933,0.001983,0.000000,0.006246,general_refresh_review,monitor,...,0,2,0.0,0.00,0.00,0.00,91,1,2929.0,new
29998,content_28b4223f4e5f,client_98a3ab7c34,29999,0.008021,0.017933,0.001983,0.000000,0.005045,general_refresh_review,monitor,...,0,1,0.0,0.00,0.00,100.00,91,1,3109.0,down


In [21]:
# Re-run the model comparison and print a clean results table.
# This uses the train/test split already created above.

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ---------- Precision@K ----------
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_idx = np.argsort(scores)[::-1][:k]

    return float(y_true[top_idx].sum() / k)


# ---------- Models ----------
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=50,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    )
}

# ---------- Baseline ----------
baseline_scores = baseline_df.set_index("content_id")[
    "baseline_refresh_score"
]

test_baseline = (
    test["content_id"]
    .map(baseline_scores)
    .fillna(0)
    .to_numpy()
)

results = []

results.append({
    "model": "Week-4 Baseline",
    "Precision@50": precision_at_k(
        test["is_declining_label"].to_numpy(),
        test_baseline,
        50
    ),
    "Accuracy": np.nan,
    "Precision": np.nan,
    "Recall": np.nan,
    "F1": np.nan,
    "ROC-AUC": np.nan,
    "Average Precision": np.nan
})

# ---------- Train models ----------
trained_models = {}
model_scores = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    trained_models[name] = model
    model_scores[name] = proba

    results.append({
        "model": name,
        "Precision@50": precision_at_k(
            y_test,
            proba,
            50
        ),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, pred, zero_division=0
        ),
        "F1": f1_score(
            y_test, pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test, proba
        ),
        "Average Precision": average_precision_score(
            y_test, proba
        )
    })

# ---------- Comparison table ----------
comparison_df = pd.DataFrame(results)

print("MODEL COMPARISON")
display(comparison_df.round(3))

# ---------- Identify best model by Precision@50 ----------
model_only = comparison_df[
    comparison_df["model"] != "Week-4 Baseline"
]

best_model_name = (
    model_only
    .sort_values("Precision@50", ascending=False)
    .iloc[0]["model"]
)

best_model = trained_models[best_model_name]
best_scores = model_scores[best_model_name]

print("\nBest learned model:", best_model_name)
print(
    "Best model Precision@50:",
    round(
        model_only.loc[
            model_only["model"] == best_model_name,
            "Precision@50"
        ].iloc[0],
        3
    )
)

MODEL COMPARISON


,model,Precision@50,Accuracy,Precision,Recall,F1,ROC-AUC,Average Precision
0,Week-4 Baseline,0.44,NaN,NaN,NaN,NaN,NaN,NaN
1,Logistic Regression,0.76,0.619,0.627,0.673,0.649,0.649,0.639
2,Decision Tree,0.68,0.624,0.649,0.615,0.632,0.655,0.623
3,Random Forest,0.44,0.626,0.620,0.741,0.675,0.660,0.620



Best learned model: Logistic Regression
Best model Precision@50: 0.76


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



I evaluate the model by looking at cases where its prediction disagrees with the observed label. False positives are pages the model ranks as likely declining when the observed label is not declining, while false negatives are declining pages that the model does not identify.

I also inspect which features the model relies on most. These are model associations within the held-out test data, not evidence that the features cause a page to decline.

The error analysis is useful because a single overall metric does not show where the model can fail or whether its strongest signals make sense for the content-refresh decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.